# Milestone 6 — RAGAS Evaluation Analysis

This notebook loads the CSV results produced by `evaluation/run_ragas.py` and visualises:
1. Aggregate metric scores (bar chart)
2. Per-question score heatmap
3. Per-pillar breakdown
4. Weakest questions — ranked by lowest combined score

> **Pre-requisite:** Run `python evaluation/run_ragas.py` with a valid `GOOGLE_API_KEY` first.

In [1]:
import glob
import os
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import pandas as pd

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'DejaVu Sans'

PROJECT_ROOT = Path('.').resolve().parent
RESULTS_DIR  = PROJECT_ROOT / 'evaluation' / 'results'
print('Results directory:', RESULTS_DIR)

Results directory: /Users/andriipikuza/Desktop/ai-exploration-sprint/02-rag-fitness-coach/evaluation/results


In [2]:
# ── Load the most recent baseline CSV ────────────────────────────────────────
csv_files = sorted(RESULTS_DIR.glob('baseline_*.csv'), reverse=True)

if not csv_files:
    print('⚠  No result files found in', RESULTS_DIR)
    print('   Run: python evaluation/run_ragas.py')
else:
    latest = csv_files[0]
    print(f'Loading: {latest.name}  ({len(csv_files)} file(s) found)')
    df = pd.read_csv(latest)
    print(df.shape)
    df.head()

⚠  No result files found in /Users/andriipikuza/Desktop/ai-exploration-sprint/02-rag-fitness-coach/evaluation/results
   Run: python evaluation/run_ragas.py


In [3]:
# ── Identify metric columns ───────────────────────────────────────────────────
METRIC_COLS = [c for c in ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
               if c in df.columns]
print('Metric columns found:', METRIC_COLS)
df[METRIC_COLS].describe().round(4)

NameError: name 'df' is not defined

## 1  Aggregate Metric Scores

In [ ]:
means = df[METRIC_COLS].mean()
labels = [m.replace('_', '\n').title() for m in METRIC_COLS]
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(labels[::-1], means.values[::-1], color=colors[::-1], height=0.55, edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, means.values[::-1]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, 1.15)
ax.axvline(0.7, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='Good threshold (0.7)')
ax.set_xlabel('Score (0–1)', fontsize=12)
ax.set_title('RAG Fitness Coach — RAGAS Baseline Evaluation', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'aggregate_scores.png', bbox_inches='tight')
plt.show()
print('Saved aggregate_scores.png')

## 2  Per-Question Score Heatmap

In [ ]:
if 'question' in df.columns:
    # Truncate long questions for display
    q_labels = [q[:55] + '…' if len(q) > 55 else q for q in df['question']]
else:
    q_labels = [f'Q{i+1}' for i in range(len(df))]

heat_data = df[METRIC_COLS].astype(float).values

fig, ax = plt.subplots(figsize=(11, max(6, len(q_labels) * 0.38)))
im = ax.imshow(heat_data, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)

ax.set_xticks(range(len(METRIC_COLS)))
ax.set_xticklabels([m.replace('_', '\n').title() for m in METRIC_COLS], fontsize=10)
ax.set_yticks(range(len(q_labels)))
ax.set_yticklabels(q_labels, fontsize=8)

# Annotate cells
for r in range(heat_data.shape[0]):
    for c in range(heat_data.shape[1]):
        val = heat_data[r, c]
        color = 'black' if 0.3 < val < 0.75 else 'white'
        ax.text(c, r, f'{val:.2f}', ha='center', va='center', fontsize=7.5, color=color)

plt.colorbar(im, ax=ax, label='Score')
ax.set_title('Per-Question RAGAS Scores', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'per_question_heatmap.png', bbox_inches='tight')
plt.show()
print('Saved per_question_heatmap.png')

## 3  Per-Pillar Breakdown

In [ ]:
if 'pillar' in df.columns:
    pillar_means = df.groupby('pillar')[METRIC_COLS].mean().round(3)
    print(pillar_means)

    pillars = pillar_means.index.tolist()
    x = np.arange(len(METRIC_COLS))
    width = 0.22
    pillar_colors = ['#4C72B0', '#DD8452', '#55A868']

    fig, ax = plt.subplots(figsize=(11, 5))
    for i, (pillar, color) in enumerate(zip(pillars, pillar_colors)):
        vals = pillar_means.loc[pillar, METRIC_COLS].values
        offset = (i - len(pillars) / 2 + 0.5) * width
        rects = ax.bar(x + offset, vals, width, label=pillar.replace('_', ' ').title(), color=color, alpha=0.85)
        ax.bar_label(rects, fmt='%.2f', fontsize=7.5, padding=2)

    ax.set_xticks(x)
    ax.set_xticklabels([m.replace('_', '\n').title() for m in METRIC_COLS], fontsize=10)
    ax.set_ylim(0, 1.2)
    ax.axhline(0.7, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Good threshold')
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('RAGAS Scores by Knowledge Pillar', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'pillar_breakdown.png', bbox_inches='tight')
    plt.show()
    print('Saved pillar_breakdown.png')
else:
    print('No pillar column found — run with full per-sample output')

## 4  Weakest Questions

In [ ]:
df['combined_score'] = df[METRIC_COLS].mean(axis=1)
worst = df.nsmallest(5, 'combined_score')[['question', 'pillar', 'combined_score'] + METRIC_COLS]

print('\n⚠  Bottom 5 questions by combined score:')
print('─' * 100)
for _, row in worst.iterrows():
    q = row['question'][:80] + ('…' if len(row['question']) > 80 else '')
    print(f"  [{row.get('pillar','?'):15s}]  Combined: {row['combined_score']:.3f}  |  {q}")
    scores = ' | '.join(f"{m.replace('_',' ')}: {row[m]:.3f}" for m in METRIC_COLS)
    print(f"                   {scores}")
    print()

## 5  Retrieval vs. Generation Analysis

**Retrieval metrics:** context_precision, context_recall  
**Generation metrics:** faithfulness, answer_relevancy

If retrieval scores are low but generation scores are high → the model is hallucinating.  
If retrieval scores are high but generation scores are low → the prompt/LLM needs tuning.

In [ ]:
retrieval_cols = [c for c in ['context_precision', 'context_recall'] if c in df.columns]
generation_cols = [c for c in ['faithfulness', 'answer_relevancy'] if c in df.columns]

retrieval_score = df[retrieval_cols].mean(axis=1) if retrieval_cols else None
generation_score = df[generation_cols].mean(axis=1) if generation_cols else None

if retrieval_score is not None and generation_score is not None:
    fig, ax = plt.subplots(figsize=(8, 6))

    pillar_color_map = {'exercise_library': '#4C72B0', 'health_issues': '#DD8452', 'nutrition': '#55A868'}
    if 'pillar' in df.columns:
        colors_scatter = [pillar_color_map.get(p, '#888888') for p in df['pillar']]
    else:
        colors_scatter = '#4C72B0'

    ax.scatter(retrieval_score, generation_score, c=colors_scatter, s=80, alpha=0.8, edgecolors='white', linewidth=0.5)

    # Add question index labels
    for i, (rx, gy) in enumerate(zip(retrieval_score, generation_score)):
        ax.annotate(f'Q{i+1}', (rx, gy), textcoords='offset points', xytext=(5, 3), fontsize=7, alpha=0.7)

    ax.axhline(0.7, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.axvline(0.7, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel('Retrieval Score (context_precision + context_recall) / 2', fontsize=11)
    ax.set_ylabel('Generation Score (faithfulness + answer_relevancy) / 2', fontsize=11)
    ax.set_title('Retrieval vs. Generation Quality', fontsize=13, fontweight='bold')
    ax.set_xlim(-0.05, 1.1)
    ax.set_ylim(-0.05, 1.1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Legend for pillars
    if 'pillar' in df.columns:
        for label, color in pillar_color_map.items():
            ax.scatter([], [], c=color, label=label.replace('_', ' ').title(), s=60)
        ax.legend(loc='lower right', fontsize=9)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'retrieval_vs_generation.png', bbox_inches='tight')
    plt.show()
    print('Saved retrieval_vs_generation.png')

print(f'\n  Mean retrieval score : {retrieval_score.mean():.3f}')
print(f'  Mean generation score: {generation_score.mean():.3f}')

## Summary & Interpretation

| Metric | Meaning | Target |
|--------|---------|--------|
| **Faithfulness** | Are answers grounded in retrieved context? High = no hallucination | ≥ 0.8 |
| **Answer Relevancy** | Does the answer address the question? | ≥ 0.8 |
| **Context Precision** | What fraction of retrieved chunks were relevant? | ≥ 0.7 |
| **Context Recall** | Does the retrieved context cover the ground-truth info? | ≥ 0.7 |

### Improvement Levers
- **Low context precision** → improve chunking strategy or embedding model
- **Low context recall** → increase `retriever_top_k`, add more knowledge sources
- **Low faithfulness** → tighten system prompt, reduce temperature
- **Low answer relevancy** → refine prompt to be more question-focused